# M.I.N.E.R.V.A. — v2.9.0

**Misinformation Identification and Navigation through Educational Reasoning and Visual Awareness**

Android-based educational game for SHS Filipino voters that teaches misinformation literacy through a neuro-symbolic credibility-scoring pipeline (RoBERTa-Tagalog + DistilBERT-multilingual + DE-GNN + Random Forest + QLattice + neuro-symbolic GPT-2 Tagalog).

**v2.8 changes vs v2.7:**
- **Pinned dependencies** including `fsspec<=2024.6.1` (fixes the `NotImplementedError: Loading a dataset cached in a LocalFileSystem` failure that broke v2.7 runs).
- **Script 01 has a 3-tier fallback** (datasets library → direct parquet → direct CSV) so dataset acquisition is robust to Hub/fsspec breakage.
- **Script 07 (Random Forest) now consumes DE-GNN output** as input, realizing the SEQUENTIAL DE-GNN→RF pipeline specified in BATB §3.5.2 (was parallel in v2.7).
- **Legacy v2.5 GPT-2 path removed** (scripts/legacy/). The `_b` neuro-symbolic path is now canonical and exclusive.
- **41 new unit tests** (27 QLattice evaluator + 14 DE-GNN graph), bringing total to **125 passing**.
- **Notebook hardening**: install cell verifies critical packages; cell 16 uses `subprocess.run + check=True` everywhere; clone cell `chdir`s out before `rmtree`.

**Architecture (per BATB §1.4 Specific Objectives):**
1. **Hybrid credibility detection** — RoBERTa + DistilBERT dual embeddings → DE-GNN graph aggregation → Random Forest classifier (sequential per §3.5.2).
2. **QLattice symbolic regression** — derives an interpretable equation AND drives rule-constrained generation via control tokens.
3. **Unity game integration** — via the curated 668-card pool + pre-pilot pack export.
4. **System evaluation** — held-out test F1, faithfulness audit, detector validation, 125 unit tests.

Open this notebook on Colab with **T4 GPU runtime**. Total time: ~50-65 min for the full architecture run.


## 1. Configuration


In [ ]:
# ============================================================
# v2.9.0 pipeline configuration
# (v2.9.0 = v2.8.7 +
#         + scripts/35 NEW: place-name pseudonymization. Audit found 4 PH cities
#           leaking through the existing person-only pseudonymizer.
#         + templates/places_blocklist.txt: 17 regions, 80+ provinces,
#           80+ cities, common landmarks. Deterministic City W/X/Y/Z mapping.
#         + templates/response_bank_v2.json: 225 hand-shaped phrases
#           (72%% diversity) covering all 12 indicators × 3 tiers × 2 verdicts.
#           Replaces v2.8.7's stub explanation block. Faithfulness audit went
#           from 87.18%% (v2.8.7) → projected 98%%+.
#         + scripts/29 update: indicator-coverage filter pre-merge. GPT-2 cards
#           are dropped if their fired indicators don't have a response-bank
#           entry, eliminating the failure mode that caused 231 mismatches.
#         + scripts/37 NEW: held-out detector evaluation on hand-labeled
#           GPT-2 cards. Replaces the tautological 'detector accuracy on the
#           pool' metric with a real generalization claim.
#         + scripts/10b update: report stamps v2.9.0 + adds an audit assertion
#           that loudly warns if percentile binning didn't activate (the
#           failure mode that left v2.8.7's GPT-2 conditioning unlearnable).
# (v2.8.7 = v2.8.6 +
#         + scripts/29 NEW: merges GPT-2 cards into the template stream so they
#           reach the pool (previous pipeline had no merge step — GPT-2 outputs
#           were stranded in jsonl files no script ever read).
#         + Persistent-generation loop: each label retries with new seeds until
#           GPT2_MIN_PROMOTED_PER_LABEL cards survive merge, capped at
#           GPT2_MAX_ATTEMPTS to prevent infinite loops.
#         + Sentence-recovery: trim mid-word truncations back to last '.', '!',
#           '?' so usable Tagalog isn't thrown away.
#         + Code remap: GPT-2 outputs Excel-style codes from JCBlaise pseudo
#           ('Candidate AA', 'Candidate DKR'); script 29 remaps them to A/B/C
#           in order-of-first-appearance so cards conform to the game's allowlist.
#         + max_new_tokens raised 120 -> 200 to reduce truncation upstream.)
# (v2.8.6 = v2.8.5 + percentile-based control-token binning in 10b
#           (so GPT-2 can actually learn graph/qlat/ensem/tier conditioning;
#           previous fixed 0.6/0.8 thresholds put 96% of training in 'high')
#         + GPT2_EPOCHS bumped 3 -> 8 (3-epoch run only converged loss 3.49)
#         + differentiated seeds for fake/real GPT-2 generation
#         + save cell now bundles models/qlattice_equation.txt + best_qlattice.json
#         + cumulative: includes v2.8.4 feyn pin + script 08 fallback)
# ============================================================

# Repo
GITHUB_USER   = "robertgeraldsenasin"
GITHUB_REPO   = "MINERVA"
GITHUB_BRANCH = "final_ver_branch"

# ----------------------------------------------------------------
# PIPELINE MODE — pick ONE of:
#
#   "templates_only"  : skip detector training AND GPT-2 generation
#                       (~3 minutes, no GPU needed). Pool comes from
#                       deterministic templates only. Auto-disables
#                       USE_GPT2.
#
#   "full_cached"     : skip detector training IF the artifacts are
#                       cached locally or in Drive; otherwise train.
#                       GPT-2 still runs to produce neuro-symbolic
#                       generations.
#
#   "full_retrain"    : ALWAYS retrain detectors AND run GPT-2
#                       neuro-symbolic generation (~50-65 min on T4).
#                       v2.8 DEFAULT — blank-slate testing realizes
#                       the FULL architecture per BATB SO 1, 2, 4
#                       every time.
# ----------------------------------------------------------------
PIPELINE_MODE = "full_retrain"

# Drive (set False to skip Drive entirely; pipeline still works,
# falls back to a downloadable zip)
USE_DRIVE        = True
DRIVE_OUTPUT_DIR = "MINERVA_outputs"

# Detector training (Mode B — used in full_* modes)
TRAIN_RUN_ID     = "v28_panel"        # tag for this training run
TRAIN_SEEDS      = "0,1,2"            # 3 seeds for time; paper used 5
TRAIN_EPOCHS     = 3                  # default per scripts/04, 05
TRAIN_MAX_LEN    = 256

# ----------------------------------------------------------------
# GPT-2 NEURO-SYMBOLIC GENERATION (BATB Specific Objective 2)
#
# When True, runs scripts 10b/11b/12b — neuro-symbolic conditioning
# that makes GPT-2 generation reflect ALL upstream signals (label +
# DE-GNN graph confidence + QLattice symbolic-equation score +
# detector ensemble + teaching tier) via 18 control tokens.
#
# This realizes the paper's "rule-constrained content generation"
# claim. Auto-disabled when PIPELINE_MODE = "templates_only".
# Requires GPU. Adds ~15-20 minutes.
# ----------------------------------------------------------------
USE_GPT2         = True
GPT2_BASE_MODEL  = "jcblaise/gpt2-tagalog"
GPT2_EPOCHS      = 8        # v2.9.0: was 3 (insufficient — 80s training, eval loss only reached 3.35)

# v2.9.0: with percentile binning verified active and response-bank coverage
# filter, expected GPT-2 promotion rate jumps from ~10%% (v2.8.7) to ~30-40%%.
# Adjust GPT2_MIN_PROMOTED_PER_LABEL up if you want a denser pool.

# v2.9.0: persistent-generation knobs. The GPT-2 path generates batches
# of cards, runs each through scoring + script 29's filters (sentence
# recovery, candidate code remap, candidate-allowlist), and retries with
# new seeds until GPT2_MIN_PROMOTED_PER_LABEL cards survive — capped at
# GPT2_MAX_ATTEMPTS so it can't loop forever.
GPT2_MIN_PROMOTED_PER_LABEL = 100   # target survivors per label
GPT2_MAX_ATTEMPTS           = 4     # max generation batches per label
GPT2_BATCH_SIZE             = 500   # cards per batch
GPT2_MAX_NEW_TOKENS         = 200   # was 120 — fewer mid-word cutoffs
GPT2_LR          = 5e-5

# Template-mode pipeline knobs (used in ALL modes)
N_PER_TEMPLATE   = 18
TARGET_POOL      = 700
DAYS_IN_DECK     = 7
CARDS_PER_DAY    = 8
MIN_CREDIBLE_PER_DAY = 3
RNG_SEED         = 1729

# Pre-pilot pack
PILOT_N          = 30
PILOT_SEED       = 1729

# v2.8 build identifier
MINERVA_VERSION  = "v2.9.0"

print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f" M.I.N.E.R.V.A. {MINERVA_VERSION} — Configuration")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  PIPELINE_MODE = {PIPELINE_MODE!r}")
print(f"  USE_GPT2      = {USE_GPT2}")
print(f"  USE_DRIVE     = {USE_DRIVE}")
print(f"  TRAIN_SEEDS   = {TRAIN_SEEDS}")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


## 2. Mount Google Drive (gracefully — skips if unavailable)


In [ ]:
DRIVE_MOUNT_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNT_OK = True
        print("✓ Drive mounted at /content/drive")
    except (ImportError, RuntimeError, ValueError) as e:
        print(f"⚠ Drive not available ({type(e).__name__}); continuing without it")
        print("  (this is normal outside Colab or in incognito)")
else:
    print("⚠ USE_DRIVE=False — skipping Drive mount")


## 3. Clone the repo


In [ ]:
import os, shutil, subprocess

REPO_PATH = "/content/MINERVA"

# CRITICAL: chdir to a known-existing parent BEFORE any rmtree.
# Colab keeps the kernel's cwd between runs. If a previous cell did
# os.chdir(REPO_PATH) and we then rmtree(REPO_PATH), the kernel ends
# up sitting in a deleted directory and `git clone` fails with
# "Unable to read current working directory".
os.chdir("/content")

if os.path.exists(REPO_PATH):
    print(f"⚠ {REPO_PATH} already exists — removing and re-cloning")
    shutil.rmtree(REPO_PATH)

clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
print(f"Cloning {clone_url} (branch: {GITHUB_BRANCH})...")
result = subprocess.run(
    ["git", "clone", "-b", GITHUB_BRANCH, "--depth", "1", clone_url, REPO_PATH],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"✗ Clone failed:\n{result.stderr}")
    raise RuntimeError("git clone failed")

os.chdir(REPO_PATH)
print(f"✓ Repo at {REPO_PATH}")
print(f"  Branch: {subprocess.check_output(['git','branch','--show-current'],text=True).strip()}")
print(f"  HEAD  : {subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip()}")


## 4. Verify required files arrived


In [ ]:
from pathlib import Path

required = [
    "scripts/candidate_config.py",            # ⭐ EDITABLE
    "scripts/30_template_scenario_generator.py",
    "scripts/31_universal_pseudonymize.py",
    "scripts/32_validate_detectors_on_templates.py",
    "scripts/40_export_pilot_pack.py",
    "scripts/26_faithfulness_audit.py",
    "scripts/21_balance_unity_cards.py",
    "scripts/23_enforce_election_theme.py",
    "scripts/24_curate_teaching_cards.py",
    "scripts/28_draw_user_deck.py",
    "scripts/minerva_candidates.py",
    "scripts/minerva_filters.py",
    "scripts/minerva_schemas.py",
    "scripts/minerva_indicators.py",
    "scripts/minerva_response_bank.py",
    "tests/test_filters.py",
    "tests/test_pilot_pack.py",
    "requirements.txt",
    "CLAUDE.md",
]

print("Required files in repo:")
all_ok = True
for f in required:
    ok = Path(f).exists()
    flag = "✓" if ok else "✗ MISSING"
    print(f"  {flag}  {f}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("Missing files — patch may not have been pushed yet. Check the branch.")

# Show current candidate config
import sys
sys.path.insert(0, "scripts")
import candidate_config
print()
print("Candidate config (edit scripts/candidate_config.py to change):")
for c in candidate_config.CANDIDATES_CONFIG:
    print(f"  {c['code']}: {candidate_config.full_name(c)} ({c['archetype']}, {c['region']})")

print()
print("✓ All v2.6-final-consolidated files present.")


## 5. Install dependencies (~30s)


In [ ]:
# v2.8 install — uses requirements.txt with proper pins.
#
# CRITICAL: do NOT subsequently install fsspec or datasets manually —
# Colab's defaults will overwrite the pinned versions and re-introduce
# the LocalFileSystem NotImplementedError that broke v2.7 runs.

import subprocess, sys

print("Installing v2.8 dependencies (this takes ~2-3 min)...")
result = subprocess.run(
    ["pip", "install", "-q", "--upgrade", "pip"],
    capture_output=False, check=False
)
result = subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=False, check=False
)

# Verify the critical packages have correct versions
print("\nVerifying versions:")
import importlib

def check(pkg, attr="__version__"):
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, attr, "?")
        print(f"  {pkg}: {v}")
        return v
    except Exception as e:
        print(f"  {pkg}: NOT INSTALLED ({e})")
        return None

fsspec_v = check("fsspec")
datasets_v = check("datasets")
transformers_v = check("transformers")
huggingface_hub_v = check("huggingface_hub")
torch_v = check("torch")
sklearn_v = check("sklearn")
feyn_v = check("feyn")

# Critical compatibility check: fsspec must be <= 2024.6.1
if fsspec_v:
    parts = [int(x) for x in fsspec_v.replace("rc", ".").split(".")[:3] if x.isdigit()]
    year, month = parts[0], parts[1] if len(parts) > 1 else 0
    if (year, month) > (2024, 6):
        print(f"\n⚠ WARNING: fsspec {fsspec_v} may be incompatible with datasets.")
        print(f"  Run this to force-fix:")
        print(f"  !pip install --force-reinstall 'fsspec>=2023.10.0,<=2024.6.1' 'datasets>=2.14,<3.0'")


# ----------------------------------------------------------------
# feyn override (per user request: no version pin, latest PyPI default)
# ----------------------------------------------------------------
# requirements.txt pins feyn to a range. The user prefers `pip install feyn`
# with no version constraint so PyPI returns the default (currently 3.5.0).
# This deliberately overrides the pinned version and is safe because feyn 3.x
# is API-stable for the QLattice usage in scripts/08.
print("\nInstalling feyn (no version pin, latest PyPI)...")
_feyn = subprocess.run(
    ["pip", "install", "-q", "feyn"],
    capture_output=True, text=True
)
if _feyn.returncode != 0:
    print(f"⚠ feyn install failed:")
    print(f"  stderr: {_feyn.stderr[-400:]}")
    print(f"  Pipeline will use sklearn LogReg fallback in scripts/08.")
else:
    # Re-import to confirm the just-installed version
    try:
        import importlib, feyn
        importlib.reload(feyn)
        print(f"  ✓ feyn {feyn.__version__} installed")
    except ImportError as e:
        print(f"  ⚠ feyn imported but failed: {e}")

print("\n✓ Install verification complete.")


## 6. Working folders


In [ ]:
for d in ["generated", "reports", "generated/decks", "reports/pilot_pack"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✓ Folders ready: generated/, reports/, generated/decks/, reports/pilot_pack/")


## 7. Run all unit tests (smoke test before pipeline)

Expected: **125 passed** (40 prior + 19 strict_allowlist + 25 neurosymbolic_corpus + 27 qlattice_evaluator + 14 degnn_graph).


In [ ]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print(result.stderr[-500:] if result.stderr else '')

# Expected: 125 passed (40 prior + 19 strict_allowlist + 25 neurosymbolic_corpus
# + 27 qlattice_evaluator + 14 degnn_graph)
if result.returncode != 0:
    raise SystemExit(f'Tests failed: {result.returncode}')
print('✓ All tests pass — proceeding to pipeline')


## 7a. Pre-flight: verify script 01 (dataset download) — v2.9.0

This is a fast (~3 second) check that runs **only** `scripts/01_download_dataset.py` and verifies the output CSV looks right. v2.9.0 changes the script's primary tier from `datasets.load_dataset()` to a direct urllib + zipfile bypass that avoids the `UnicodeDecodeError` (gzip magic 0x8b) that broke v2.8 / v2.8.1.

**If this cell fails, do NOT continue to section 7b** — the rest of the pipeline depends on the dataset being downloaded successfully. See the printed error and the manual fallback instructions in the script's FATAL block.


In [ ]:
# Pre-flight: run script 01 and validate its output.
import subprocess, sys
from pathlib import Path
import pandas as pd

if PIPELINE_MODE == "templates_only":
    print("⏭️  Skipping pre-flight (templates_only mode — no dataset needed).")
else:
    print("Running scripts/01_download_dataset.py ...")
    res = subprocess.run([sys.executable, "scripts/01_download_dataset.py"],
                         capture_output=False)
    if res.returncode != 0:
        raise SystemExit(
            f"✗ Script 01 failed with exit code {res.returncode}. "
            "Read the FATAL block above for diagnosis. Do NOT proceed to 7b."
        )

    csv_path = Path("data/raw/jcblaise.csv")
    assert csv_path.exists(), f"✗ Expected output {csv_path} was not created"

    df = pd.read_csv(csv_path)
    assert "text" in df.columns or "article" in df.columns, \
        f"✗ Missing text column. Got: {list(df.columns)}"
    assert "label" in df.columns, \
        f"✗ Missing label column. Got: {list(df.columns)}"

    n_rows = len(df)
    label_counts = dict(df["label"].value_counts().sort_index())
    print(f"\n✓ Pre-flight PASSED:")
    print(f"  rows         : {n_rows}")
    print(f"  columns      : {list(df.columns)}")
    print(f"  label dist.  : {label_counts}")
    print(f"  expected ~3,206 rows, roughly balanced 0/1")
    if not (3000 <= n_rows <= 3500):
        print(f"  ⚠  Row count {n_rows} is outside expected range "
              f"— inspect the CSV before continuing.")


## 7b. **Mode B foundation** — Detector training pipeline (scripts 01-17)

This block runs the upstream stages that **produce the neuro-symbolic signals** the rest of the pipeline depends on:

| # | Script | What it does | Output |
|---|---|---|---|
| 01 | `01_download_dataset.py` | Pull JCBlaise from HuggingFace | `data/raw/jcblaise.csv` |
| 02 | `02_prepare_dataset.py` | Clean, normalize | `data/processed/jcblaise_clean.csv` |
| 03 | `03_split_dataset.py` | 70/15/15 train/val/test | `data/processed/{train,val,test}.csv` |
| 04 | `04_train_robertaMINERVA.py` | Fine-tune RoBERTa-Tagalog | `models/roberta_finetuned/` |
| 05 | `05_train_distilbertMINERVA.py` | Fine-tune DistilBERT-multilingual | `models/distilbert_multilingual_finetuned/` |
| 17 | `17_run_5seeds_detectors.py` | Multi-seed averaging (paper rigor) | `models/<task>/run_<id>/seed_<n>/` |
| 06 | `06_extract_features.py` | CLS embeddings + PCA | `data/features/train_tabular.csv` |
| 07 | `07_train_random_forest.py` | RF baseline | `models/random_forest.joblib` |
| 08 | `08_train_qlattice.py` | Symbolic regression | **`models/qlattice_equation.txt`** |
| 09 | `09_train_degnn.py` | Dual-Embedding GNN | **`data/features/degnn_preds.csv`** |
| 15 | `15_evaluate_detectors.py` | Test-set metrics | `logs/eval_detectors_report.json` |

**This is the foundation of M.I.N.E.R.V.A.'s neuro-symbolic claim.** Without these artifacts, scripts 10b / 11b / 12b have no QLattice equation or DE-GNN confidence to condition on.

**Behavior by mode:**
- `templates_only`: skipped entirely.
- `full_cached`: skipped IF artifacts exist on Drive; otherwise trained.
- `full_retrain`: always retrained.

Time on Colab T4: ~25-40 minutes for a fresh full run.


In [ ]:
# Detector training — gated by PIPELINE_MODE
import os, shutil, subprocess, sys
from pathlib import Path

REQUIRED_ARTIFACTS = [
    "data/processed/train.csv",
    "data/processed/val.csv",
    "data/processed/test.csv",
    "data/features/train_tabular.csv",
    "data/features/val_tabular.csv",
    "data/features/test_tabular.csv",
    "data/features/degnn_preds.csv",
    "models/qlattice_equation.txt",
    "models/roberta_finetuned/config.json",
    "models/distilbert_multilingual_finetuned/config.json",
]

# Clear stale env from any previous run
os.environ.pop("_NEED_TRAIN", None)

def _run_step(cmd, expected_outputs):
    """Run a pipeline script and verify it produced expected outputs.

    v2.9.0: Streams output line-by-line and NEVER raises SystemExit.
    SystemExit triggers a Colab/IPython 3.11 bug that corrupts traceback
    rendering with: "AttributeError: 'tuple' object has no attribute 'f_lineno'"
    and hides the actual error from the script. We use Popen + line streaming
    + RuntimeError instead, so the real failure is always visible.

    cmd: list of strings, e.g. ["python", "scripts/01_download_dataset.py"]
    expected_outputs: list of file paths that must exist after the script runs
    """
    print(f"  ▶ Running: {' '.join(cmd)}", flush=True)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge so order is preserved
        text=True,
        bufsize=1,                  # line-buffered
    )
    output_lines = []
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            output_lines.append(line)
        proc.wait()
    except KeyboardInterrupt:
        proc.kill()
        proc.wait()
        raise

    rc = proc.returncode
    if rc != 0:
        print(f"\n{'='*60}")
        print(f"✗ STEP FAILED with exit code {rc}: {' '.join(cmd)}")
        print(f"{'='*60}")
        if output_lines:
            print("Last 40 lines of output (look here for the actual error):")
            print("-" * 60)
            for line in output_lines[-40:]:
                print(line, end="")
            print("-" * 60)
        else:
            print("(no output captured — script crashed before writing anything)")
        # RuntimeError, NOT SystemExit. SystemExit triggers the Colab/IPython
        # 3.11 traceback-rendering bug that hides the real error.
        raise RuntimeError(
            f"Pipeline aborted at: {cmd[1] if len(cmd) > 1 else cmd[0]} "
            f"(exit {rc}). See output above for diagnosis."
        )

    missing = [p for p in expected_outputs if not Path(p).exists()]
    if missing:
        print(f"\n✗ STEP COMPLETED BUT EXPECTED OUTPUTS MISSING:")
        for m in missing:
            print(f"    - {m}")
        raise RuntimeError(f"Output verification failed for: {' '.join(cmd)}")

    if expected_outputs:
        print(f"  ✓ Verified {len(expected_outputs)} output(s)")

def _try_restore_from_drive():
    """If Drive is mounted and a previous run cached artifacts, restore them."""
    if not DRIVE_MOUNT_OK:
        return False
    cache_dir = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / "detector_artifacts"
    if not cache_dir.exists():
        return False
    print(f"  Restoring detector artifacts from Drive: {cache_dir}")
    for sub in ("data", "models", "logs"):
        src = cache_dir / sub
        if src.exists():
            dst = Path(sub)
            dst.mkdir(parents=True, exist_ok=True)
            for item in src.rglob("*"):
                if item.is_file():
                    rel = item.relative_to(src)
                    target = dst / rel
                    target.parent.mkdir(parents=True, exist_ok=True)
                    if not target.exists():
                        shutil.copy2(item, target)
    return True

def _save_to_drive():
    """Cache artifacts to Drive for the next session."""
    if not DRIVE_MOUNT_OK:
        return
    cache_dir = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / "detector_artifacts"
    cache_dir.mkdir(parents=True, exist_ok=True)
    for sub in ("data", "models", "logs"):
        if Path(sub).exists():
            shutil.copytree(sub, cache_dir / sub, dirs_exist_ok=True)
    print(f"  ✓ Cached artifacts to Drive: {cache_dir}")

def _artifacts_present():
    return all(Path(p).exists() for p in REQUIRED_ARTIFACTS)

if PIPELINE_MODE == "templates_only":
    print("PIPELINE_MODE = templates_only — skipping detector training.")
    print("  Mode A (templates) does not require detector artifacts.")
    print("  GPT-2 path will be auto-disabled.")
    USE_GPT2 = False
elif PIPELINE_MODE == "full_cached":
    print("PIPELINE_MODE = full_cached — checking for cached artifacts...")
    if _artifacts_present():
        print("  ✓ All required artifacts present locally; skipping training.")
    else:
        if _try_restore_from_drive() and _artifacts_present():
            print("  ✓ Restored artifacts from Drive; skipping training.")
        else:
            print("  ⏵ No cache — running full detector training pipeline below.")
            os.environ["_NEED_TRAIN"] = "1"
elif PIPELINE_MODE == "full_retrain":
    print("PIPELINE_MODE = full_retrain — forcing retrain.")
    os.environ["_NEED_TRAIN"] = "1"
else:
    raise ValueError(f"Unknown PIPELINE_MODE: {PIPELINE_MODE!r}")


### 7b.1 Data preparation (scripts 01 / 02 / 03)

Downloads JCBlaise, normalizes text, splits 70/15/15.


In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/01_download_dataset.py"],
              expected_outputs=["data/raw/jcblaise.csv"])
    # Script 02 writes data/processed/corpus.csv (NOT jcblaise_clean.csv —
    # that was a v2.8 path-mismatch bug fixed in v2.9.0).
    _run_step(["python", "scripts/02_prepare_dataset.py"],
              expected_outputs=["data/processed/corpus.csv"])
    _run_step(["python", "scripts/03_split_dataset.py"],
              expected_outputs=["data/processed/train.csv",
                                "data/processed/val.csv",
                                "data/processed/test.csv"])
    print("✓ Data prepared.")
else:
    print("⏭  Skipping data prep (cached or templates_only).")


### 7b.2 Train RoBERTa-Tagalog + DistilBERT-multilingual (scripts 04 / 05 / 17)

RoBERTa-Tagalog and DistilBERT-multilingual are fine-tuned for binary REAL/FAKE classification. Script 17 wraps them with the multi-seed protocol from the paper (3 seeds for Colab time-budget; the published paper used 5).

Frozen test-set metrics from the paper: RoBERTa F1 = 95.6%, DistilBERT F1 = 91.0%.


In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    # Script 17 wraps scripts 04 and 05 with the multi-seed protocol from the paper
    _run_step(
        ["python", "scripts/17_run_5seeds_detectors.py",
         "--run_id", TRAIN_RUN_ID,
         "--seeds", TRAIN_SEEDS],
        expected_outputs=[]  # produces models/<task>/run_<id>/seed_<n>/ — variable structure
    )
    # Pick the best seed and copy to canonical location (scripts/04 and 05 expected paths)
    try:
        _run_step(
            ["python", "scripts/export_best_detectors_by_val.py",
             "--run_id", TRAIN_RUN_ID],
            expected_outputs=["models/roberta_finetuned/config.json",
                              "models/distilbert_multilingual_finetuned/config.json"]
        )
    except SystemExit:
        print("  (export_best_detectors_by_val.py not found or failed — falling back)")
        # Fallback: train each detector once at canonical location
        _run_step(["python", "scripts/04_train_robertaMINERVA.py"],
                  expected_outputs=["models/roberta_finetuned/config.json"])
        _run_step(["python", "scripts/05_train_distilbertMINERVA.py"],
                  expected_outputs=["models/distilbert_multilingual_finetuned/config.json"])
    print("✓ Detectors trained.")
else:
    print("⏭  Skipping detector training (cached or templates_only).")


### 7b.3 Extract features (script 06)

Pools the [CLS] embeddings from RoBERTa and DistilBERT, applies PCA per encoder, and writes the tabular feature CSVs (`r_pca_*`, `d_pca_*`, `p_roberta_fake`, `p_distil_fake`).


In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/06_extract_features.py"],
              expected_outputs=["data/features/train_tabular.csv",
                                "data/features/val_tabular.csv",
                                "data/features/test_tabular.csv"])
    print("✓ Features extracted.")
else:
    print("⏭  Skipping feature extraction (cached or templates_only).")


### 7b.4 Train QLattice symbolic regression (script 08)

Produces the **interpretable equation** that drives M.I.N.E.R.V.A.'s explainability claim. Output: `models/qlattice_equation.txt` — a closed-form expression like `logreg(0.371093 * dpca1 + ...)` evaluable on any feature row.

This equation is **also used as a generation conditioning signal** by script 10b (the v2.6.final neuro-symbolic GPT-2 path).


In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/08_train_qlattice.py"],
              expected_outputs=["models/qlattice_equation.txt"])
    print("✓ QLattice trained.")
    print("\n  Equation:")
    print("  " + Path("models/qlattice_equation.txt").read_text(encoding="utf-8").strip())
else:
    print("⏭  Skipping QLattice training (cached or templates_only).")
    if Path("models/qlattice_equation.txt").exists():
        print("\n  Cached QLattice equation:")
        print("  " + Path("models/qlattice_equation.txt").read_text(encoding="utf-8").strip())


### 7b.5 Train Dual-Embedding GNN (script 09)

Trains the DE-GNN on the dual-embedding graph (kNN over RoBERTa + DistilBERT features). Outputs `data/features/degnn_preds.csv` with `p_degnn_fake` per row.

Frozen test-set metric from the paper: DE-GNN F1 = 95.8%. The DE-GNN's per-row confidence is the third conditioning signal for the v2.6.final GPT-2 path.


In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/09_train_degnn.py"],
              expected_outputs=["data/features/degnn_preds.csv"])
    print("✓ DE-GNN trained.")
else:
    print("⏭  Skipping DE-GNN training (cached or templates_only).")


### 7b.6 Held-out test-set evaluation (script 15)

Runs the trained models against the held-out test split for the panel-defensible numbers (NOT just the synthetic-score validation that script 32 does later).


In [ ]:
EVAL_REPORT = "logs/eval_detectors_report.json"
TEST_CSV = "data/processed/test.csv"

if os.environ.get("_NEED_TRAIN") == "1":
    # Verify upstream artifacts exist before invoking script 15
    if not Path(TEST_CSV).exists():
        print(f"⚠  Cannot run held-out evaluation: {TEST_CSV} is missing.")
        print(f"    This means scripts 01/02/03 (data prep) didn't complete successfully.")
        print(f"    Re-run cell 7b.1 above, or set PIPELINE_MODE='templates_only'.")
    else:
        _run_step(
            ["python", "scripts/15_evaluate_detectors.py",
             "--test_csv", TEST_CSV,
             "--out_json", EVAL_REPORT],
            expected_outputs=[EVAL_REPORT]
        )
        print("✓ Detector evaluation complete.")
        import json as _json
        rep = _json.load(open(EVAL_REPORT))
        print(_json.dumps(rep.get("summary", rep), indent=2))
        # Cache to Drive for next session
        _save_to_drive()
elif PIPELINE_MODE != "templates_only" and Path(EVAL_REPORT).exists():
    print("Cached evaluation report:")
    import json as _json
    rep = _json.load(open(EVAL_REPORT))
    print(_json.dumps(rep.get("summary", rep), indent=2))
else:
    print("⏭  Skipping evaluation (templates_only).")


### 7b ✓ checkpoint — neuro-symbolic signals ready

If you ran in `full_cached` or `full_retrain` mode, you should now have:
- Trained detectors at `models/roberta_finetuned/` and `models/distilbert_multilingual_finetuned/`
- The QLattice equation at `models/qlattice_equation.txt`
- DE-GNN predictions at `data/features/degnn_preds.csv`
- Detector evaluation at `logs/eval_detectors_report.json`

These are the upstream signals that scripts 10b / 11b / 12b condition GPT-2 generation on. Without them, the GPT-2 path falls back to label-only conditioning (still works, just doesn't use the neuro-symbolic claim).

**Now we proceed to the card-generation pipeline (which works in all three modes).**


## 8. **Pipeline step 1** — Generate template cards (script 30)

The 18 templates produce 900 raw cards (18 × 18 per-template × 3 archetypes when applicable).


In [ ]:
import os
os.environ["N_PER_TEMPLATE"] = str(N_PER_TEMPLATE)

!python scripts/30_template_scenario_generator.py \
    --out_file generated/template_cards.json \
    --n_per_template $N_PER_TEMPLATE \
    --report_out reports/template_gen.json


## 8b. (Optional Mode B) Neuro-symbolic GPT-2 augmentation — only if `USE_GPT2 = True`

This runs the v2.6.final neuro-symbolic conditioning pipeline:

  1. **Script 10b** — joins detector probabilities, DE-GNN confidence, and the QLattice equation into a 5-token-conditioned corpus (`<|label=...|>`, `<|graph=...|>`, `<|qlat=...|>`, `<|ensem=...|>`, `<|tier=...|>`)
  2. **Script 11b** — fine-tunes `jcblaise/gpt2-tagalog` with the 18 special tokens; the first V_base embedding rows remain byte-identical to the pretrained checkpoint, only the 18 new rows are randomly initialized
  3. **Script 12b** — generates with all five conditioning tokens prepended

**This is the paper's neuro-symbolic architecture (BATB §1.4 Specific Objective 2 + §2.6.3) realized in code.** The detector ensemble, DE-GNN graph confidence, and QLattice equation become *generation conditioning signals* — not just post-hoc filters.

**Prerequisites:** scripts 04/05/06/08/09 must have run first to produce the upstream signals. If `USE_GPT2 = False`, this cell is skipped and the pipeline runs in template-only mode (which is enough for the demo and for the 668-card pool).

Citations: Keskar et al. 2019 (CTRL); Christensen et al. 2022 (QLattice); Hamilton et al. 2017 (GraphSAGE); Bhuyan et al. 2024 (neuro-symbolic AI).


In [ ]:
import json

if USE_GPT2:
    print("=" * 60)
    print("v2.9.0: NEURO-SYMBOLIC GPT-2 PATH (BATB SO 2)")
    print("=" * 60)

    needed = [
        "data/processed/train.csv",
        "data/processed/val.csv",
        "data/features/train_tabular.csv",
        "data/features/val_tabular.csv",
        "data/features/degnn_preds.csv",
        "models/qlattice_equation.txt",
    ]
    missing = [p for p in needed if not Path(p).exists()]
    if missing:
        print(f"⚠ Missing upstream artifacts: {missing}")
        print(f"  Section 7b above produces these. Set PIPELINE_MODE='full_*'")
        print(f"  Skipping GPT-2 path — pipeline continues with templates only.")
    else:
        # ----------------------------------------------------------------
        # 10b: build neuro-symbolic corpus (percentile binning by default)
        # ----------------------------------------------------------------
        _run_step(
            ["python", "scripts/10b_prepare_gpt2_neurosymbolic.py",
             "--train_csv", "data/processed/train.csv",
             "--val_csv", "data/processed/val.csv",
             "--train_features", "data/features/train_tabular.csv",
             "--val_features", "data/features/val_tabular.csv",
             "--degnn_preds", "data/features/degnn_preds.csv",
             "--qlattice_equation", "models/qlattice_equation.txt",
             "--out_dir", "data/gpt2_neurosymbolic",
             "--report_out", "reports/gpt2_neurosymbolic_corpus.json"],
            expected_outputs=["data/gpt2_neurosymbolic/train.txt",
                              "data/gpt2_neurosymbolic/val.txt"]
        )

        # ----------------------------------------------------------------
        # 11b: fine-tune
        # ----------------------------------------------------------------
        _run_step(
            ["python", "scripts/11b_train_gpt2_neurosymbolic.py",
             "--corpus_dir", "data/gpt2_neurosymbolic",
             "--base_model", GPT2_BASE_MODEL,
             "--out_dir", "models/gpt2_tagalog_neurosymbolic",
             "--epochs", str(GPT2_EPOCHS),
             "--lr", str(GPT2_LR)],
            expected_outputs=["models/gpt2_tagalog_neurosymbolic/config.json"]
        )

        # ----------------------------------------------------------------
        # 12b + 13: PERSISTENT GENERATION LOOP (v2.9.0)
        # Generate a batch, score it, count how many would survive merge,
        # retry with a new seed if not enough — up to GPT2_MAX_ATTEMPTS.
        # ----------------------------------------------------------------
        def _count_promoted_for_label(accumulator_path, label):
            """Run script 29 in dry mode against `accumulator_path` only
            (empty templates, empty other-label) and return promoted count."""
            empty_other = Path("/tmp/_empty.jsonl"); empty_other.touch()
            empty_templates = Path("/tmp/_empty_templates.json")
            empty_templates.write_text("[]")
            dry_out = Path("/tmp/_dry_merged.json")
            dry_report = Path("/tmp/_dry_report.json")
            cmd = ["python", "scripts/29_merge_gpt2_into_pool.py",
                   "--templates", str(empty_templates),
                   "--gpt2_fake", str(accumulator_path) if label == "fake" else str(empty_other),
                   "--gpt2_real", str(accumulator_path) if label == "real" else str(empty_other),
                   "--out", str(dry_out),
                   "--report_out", str(dry_report)]
            _run_step(cmd, expected_outputs=[str(dry_report)])
            rep = json.loads(dry_report.read_text())
            return int(rep.get("gpt2_promoted_by_label", {}).get(label, 0))

        def _generate_until_target(label, base_seed):
            accumulator = Path(f"generated/gpt2_neuro_final_{label}.jsonl")
            if accumulator.exists():
                accumulator.unlink()
            accumulator.touch()

            n_promoted = 0
            for attempt in range(GPT2_MAX_ATTEMPTS):
                seed = base_seed + attempt
                raw = f"generated/_a{attempt}_{label}_raw.jsonl"
                scored = f"generated/_a{attempt}_{label}_scored.jsonl"
                final = f"generated/_a{attempt}_{label}_final.jsonl"

                _run_step(
                    ["python", "scripts/12b_generate_gpt2_neurosymbolic.py",
                     "--target", label, "--n", str(GPT2_BATCH_SIZE),
                     "--seed", str(seed),
                     "--max_new_tokens", str(GPT2_MAX_NEW_TOKENS),
                     "--gen_model_dir", "models/gpt2_tagalog_neurosymbolic",
                     "--graph", "high", "--qlat", "high", "--ensem", "high",
                     "--tier", "novice",
                     "--out", raw],
                    expected_outputs=[raw]
                )
                _run_step(
                    ["python", "scripts/13_score_generated_with_qlattice.py",
                     "--in_jsonl", raw, "--target", label,
                     "--out_scored", scored, "--out_final", final],
                    expected_outputs=[final]
                )

                with open(final, "r", encoding="utf-8") as fi, \
                     open(accumulator, "a", encoding="utf-8") as fo:
                    fo.writelines(fi)

                n_promoted = _count_promoted_for_label(accumulator, label)
                print(f"  [{label}] attempt {attempt+1}/{GPT2_MAX_ATTEMPTS} "
                      f"(seed {seed}): {n_promoted} promoted so far "
                      f"(target {GPT2_MIN_PROMOTED_PER_LABEL})")

                if n_promoted >= GPT2_MIN_PROMOTED_PER_LABEL:
                    print(f"  ✓ [{label}] target reached, stopping retries.")
                    break
            else:
                print(f"  ⚠ [{label}] max attempts reached. "
                      f"Got {n_promoted} promoted (target was {GPT2_MIN_PROMOTED_PER_LABEL}).")
            return n_promoted

        print()
        print("─" * 60)
        print("Persistent generation: target ≥{} per label, max {} attempts."
              .format(GPT2_MIN_PROMOTED_PER_LABEL, GPT2_MAX_ATTEMPTS))
        print("─" * 60)

        n_fake = _generate_until_target("fake", base_seed=13)
        n_real = _generate_until_target("real", base_seed=27)

        print()
        print(f"✓ Neuro-symbolic GPT-2 generation complete.")
        print(f"  fake: {n_fake} cards in generated/gpt2_neuro_final_fake.jsonl")
        print(f"  real: {n_real} cards in generated/gpt2_neuro_final_real.jsonl")
        print(f"  These will be merged with template cards by script 29 (next cell).")
else:
    print("⏭  Skipping GPT-2 path (USE_GPT2=False).")


## 8b.5 Merge GPT-2 cards into the template stream (v2.9.0)

The GPT-2 path produced `generated/gpt2_neuro_final_*.jsonl`. Until v2.9.0, no script ever merged those into the template stream that feeds pseudonymize → theme → balance → curate → strict allowlist. Script 29 closes that gap. It also:

1. **Recovers** mid-word truncations by trimming back to the last `.`/`?`/`!`
2. **Remaps** Excel-style candidate codes (e.g. `Candidate DKR`) back to A/B/C in order of first appearance — these codes come from JCBlaise pseudonymization (`minerva_privacy._index_to_code` is A..Z..AA..AZ..AAA..) and are valid in the training corpus, but the game only allows A/B/C
3. **Drops** GPT-2 cards with >3 distinct candidate entities (no clean A/B/C slot)
4. **Drops** GPT-2 cards with degenerate truncation (`empty`, `too_short`)
5. **Transforms** surviving GPT-2 cards to template-card schema using the model's own real-valued p_fake, detector scores, and named features (not synthetic stubs)
6. **Writes** `generated/template_plus_gpt2_cards.json` — the input for the pseudonymize cell
7. **Provenance** is preserved: every promoted card has `provenance.source = 'gpt2_neurosymbolic'`, so the panel can audit which cards were template-derived vs GPT-2-generated.


In [ ]:
if USE_GPT2 and Path('generated/gpt2_neuro_final_fake.jsonl').exists():
    _run_step(
        ['python', 'scripts/29_merge_gpt2_into_pool.py',
         '--templates', 'generated/template_cards.json',
         '--gpt2_fake', 'generated/gpt2_neuro_final_fake.jsonl',
         '--gpt2_real', 'generated/gpt2_neuro_final_real.jsonl',
         '--out', 'generated/template_plus_gpt2_cards.json',
         '--report_out', 'reports/merge_gpt2_into_pool.json',
         '--require_at_least', str(GPT2_MIN_PROMOTED_PER_LABEL * 2)],
        expected_outputs=['generated/template_plus_gpt2_cards.json',
                          'reports/merge_gpt2_into_pool.json']
    )
    _MERGED_INPUT_PATH = 'generated/template_plus_gpt2_cards.json'
else:
    print('⏭  GPT-2 outputs not present — pipeline continues with templates only.')
    _MERGED_INPUT_PATH = 'generated/template_cards.json'

print(f'\nNext stage will use: {_MERGED_INPUT_PATH}')


## 9. **Pipeline step 2** — Universal pseudonymization (script 31)

Catches any real-name leaks in the cards. For template-generated cards, this is
mostly a no-op (templates are clean by construction), but it stays in the pipeline
to handle Mode-B GPT-2 output if used.


In [ ]:
# v2.9.0: read merged file (templates + GPT-2 promotions) if it exists,
# else fall back to template-only.
_in = _MERGED_INPUT_PATH if '_MERGED_INPUT_PATH' in dir() else 'generated/template_cards.json'
!python scripts/31_universal_pseudonymize.py \
    --in_file {_in} \
    --out_file generated/cards_pseudo.json \
    --report_out reports/pseudo.json


## 9b. Place-name pseudonymization (v2.9.0)

The v2.8.7 audit found 4 Philippine city names leaking into the strict allowlist (Lapu-Lapu City ×5, Baguio City ×4, Pasay City ×1, Metropolitan Manila ×2). These bypass script 31's NER-based pseudonymizer because place names need a different mechanism (curated blocklist).

Script 35 adds a second pseudonymization pass that catches geographic entities. Output goes to `generated/cards_pseudo_places.json`, which becomes the input for the theme-enforcement step.


In [ ]:
!python scripts/35_pseudonymize_places.py \
    --in_file generated/cards_pseudo.json \
    --out_file generated/cards_pseudo_places.json \
    --blocklist templates/places_blocklist.txt \
    --report_out reports/pseudo_places.json


## 10. **Pipeline step 3** — Balance verdicts and candidates (script 21)


In [ ]:
os.environ["TARGET_POOL"] = str(TARGET_POOL)

!python scripts/21_balance_unity_cards.py \
    --in_file generated/cards_pseudo_places.json \
    --out_file generated/balanced.json \
    --target_total $TARGET_POOL \
    --report_out reports/balance.json


## 11. **Pipeline step 4** — Election-theme filter (script 23)

Templates are on-theme by construction, so accept rate should be ~95%+.
The few rejects are usually no-candidate-mention edge cases.


In [ ]:
!python scripts/23_enforce_election_theme.py \
    --in_file generated/balanced.json \
    --out_file generated/themed.json \
    --report_out reports/theme.json \
    --rejection_log reports/theme_rej.jsonl


## 12. **Pipeline step 5** — Curate teaching pool (script 24)

Enforces the credible-card-per-day quota and stratifies by tactic/tier.


In [ ]:
os.environ["DAYS_IN_DECK"] = str(DAYS_IN_DECK)
os.environ["CARDS_PER_DAY"] = str(CARDS_PER_DAY)
os.environ["MIN_CREDIBLE_PER_DAY"] = str(MIN_CREDIBLE_PER_DAY)
os.environ["RNG_SEED"] = str(RNG_SEED)

!python scripts/24_curate_teaching_cards.py \
    --in_file generated/themed.json \
    --out_file generated/unity_cards_pool.json \
    --reject_out generated/pool_rej.json \
    --report_out reports/pool.json \
    --target_pool_size $TARGET_POOL \
    --days $DAYS_IN_DECK --cards_per_day $CARDS_PER_DAY \
    --min_credible_per_day $MIN_CREDIBLE_PER_DAY \
    --seed $RNG_SEED


## 13. **Pipeline step 6** — Draw per-user decks (script 28)

Eight demo players: alice, bob, charlie, diana, erika, fiona, greg, hana.
Pairwise overlap target: < 15%.


In [ ]:
!python scripts/28_draw_user_deck.py \
    --pool_file generated/unity_cards_pool.json \
    --out_dir generated/decks \
    --user_ids "alice,bob,charlie,diana,erika,fiona,greg,hana" \
    --report_out reports/draw.json


## 14. **Pipeline step 7** — Faithfulness audit (script 26)

Every card's explanation phrases are re-checked against its `fired_indicators`.
Target: **100% (668/668 pass)**.


In [ ]:
!python scripts/26_faithfulness_audit.py \
    --in_file generated/unity_cards_pool.json \
    --report_out reports/faith.json


## 15. **Pipeline step 8** — Detector validation (script 32, NEW)

Validates the synthetic detector scores against gold verdicts. With
`uncertain_band=0.05`, all 4 detectors should hit 100% accuracy on the template pool.

Note: this validates **synthetic** scores from script 30, not a fresh inference
from the trained models. For full ground-truth validation, run script 15 against
a held-out test split (see `HANDOFF.md` P1.1).


In [ ]:
!python scripts/32_validate_detectors_on_templates.py \
    --pool_file generated/unity_cards_pool.json \
    --report_out reports/det.json \
    --markdown_out reports/det.md


## 16. **Pipeline step 9** — STRICT ALLOWLIST ENFORCER (script 33, NEW v2.6.final)

This is the closure on the user-reported issue: variety of real political names
appearing in cards causing legality and pedagogical confusion. Script 33 audits
every card text and **rejects** any card that contains person-like names not on the
three-candidate allowlist (Aurelia Santos / Bruno Villanueva / Celia Navarro).

The check uses three passes:
1. Structural patterns (title+name, "according to", said-surname, etc.)
2. Multiword capitalized spans
3. Direct blocklist scan against names from `templates/jcblaise_real_names_blocklist.txt`

Backed by Pilan et al. (2022) *Computational Linguistics* on the TAB benchmark for text
anonymization, which establishes multi-pass enforcement as the recommended standard.

**Mode `reject`** drops leaky cards (recommended for game export).
**Mode `redact`** replaces foreign names with `[Iba pang tao]` and keeps the card.

After this stage, the pool is **guaranteed** three-candidate-only.


In [ ]:
import os

# Use the candidate config (single source of truth)
CANDIDATE_PROFILES = "scripts/candidate_config.py"
BLOCKLIST = "templates/jcblaise_real_names_blocklist.txt"

print(f"Using candidate profiles: {CANDIDATE_PROFILES}")
print(f"Using blocklist file:     {BLOCKLIST}")
print()

# Run in REJECT mode - strictest, recommended for game export.
!python scripts/33_strict_name_allowlist.py \
    --in_file generated/unity_cards_pool.json \
    --candidate_profiles {CANDIDATE_PROFILES} \
    --blocklist_file {BLOCKLIST} \
    --out_file generated/unity_cards_pool_strict.json \
    --report_out reports/strict_allowlist.json \
    --mode reject

# Print key numbers from the report
import json as _json
_r = _json.load(open("reports/strict_allowlist.json"))
print()
print(f"Strict allowlist pass rate : {_r['pass_rate_pct']}%")
print(f"  Cards in    : {_r['n_input_cards']}")
print(f"  Cards out   : {_r['n_kept']}")
print(f"  Cards rejected : {_r['n_rejected']}")
if _r.get('top_foreign_names'):
    print(f"  Top foreign names found:")
    for x in _r['top_foreign_names'][:5]:
        print(f"    - {x['name']!r} ({x['reason']}) - {x['count']}x")
else:
    print(f"  No foreign names found - pool is clean!")


## 11b. Held-out detector evaluation (v2.9.0)

Replaces the v2.8.7 tautology in `reports/det.json` (which evaluated detectors on the same cards they were used to score) with a real off-distribution generalization test against hand-labeled cards in `templates/holdout_gpt2_labeled.csv`.

**Workflow:** the team should hand-label the 50-card holdout *before* running this cell. If `true_label` is empty, the script reports class balance and exits cleanly without F1.


In [ ]:
from pathlib import Path
_holdout = Path('templates/holdout_gpt2_labeled.csv')
if _holdout.exists():
    !python scripts/37_holdout_detector_eval.py \
        --holdout templates/holdout_gpt2_labeled.csv \
        --report_out reports/holdout_detector_eval.json
else:
    print('⏭  No holdout CSV found at', _holdout)
    print('   See templates/holdout_gpt2_labeled.README.md for the labeling protocol.')


## 16. **Pipeline step 9** — Pre-pilot pack (script 40, NEW P1.2)

Generates a 30-card pack stratified across verdict, tier, candidate, and tactic.
Three outputs in `reports/pilot_pack/`:

- `printable_card_pack.html` — A4 print CSS, one card per page (open and "Print to PDF")
- `questionnaire.md` — 5 questions per card, ready to paste into Google Forms
- `answer_key.md` — gold verdict + DEPICT indicators for scoring


In [ ]:
os.environ["PILOT_N"] = str(PILOT_N)
os.environ["PILOT_SEED"] = str(PILOT_SEED)

!python scripts/40_export_pilot_pack.py \
    --pool_file generated/unity_cards_pool_strict.json \
    --out_dir reports/pilot_pack \
    --n $PILOT_N --seed $PILOT_SEED


## 17. Final dashboard — verified metrics

Reads every report file and prints the consolidated check.


In [ ]:
import json, glob

def safe_load(p):
    try:
        return json.load(open(p, encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None

print("=" * 70)
print("M.I.N.E.R.V.A. v2.6-final-consolidated + P1.2 — END-TO-END METRICS")
print("=" * 70)

pool = safe_load("generated/unity_cards_pool.json")
if pool:
    meta = pool.get("_metadata", {})
    print(f"\nPool size            : {meta.get('pool_size')} cards")
    print(f"  REAL/FAKE/UNCERTAIN : {meta.get('real_count')}/{meta.get('fake_count')}/{meta.get('uncertain_count')}")
    print(f"  Candidates          : {meta.get('candidate_distribution')}")
    print(f"  Tier distribution   : {meta.get('tier_distribution')}")
    ind = meta.get("indicator_coverage", {})
    print(f"  Indicator coverage  : {len([k for k,v in ind.items() if v > 0])}/12  -- {sorted(ind.keys())}")

faith = safe_load("reports/faith.json")
if faith:
    print(f"\nFaithfulness audit    : {faith.get('pass_rate')}%  ({faith.get('passed')}/{faith.get('total_audited')})")

draw = safe_load("reports/draw.json")
if draw:
    po = draw.get("pairwise_overlap", {})
    if po:
        print(f"Pairwise overlap      : mean {po.get('mean_overlap_pct')}%  range [{po.get('min_overlap_pct')}, {po.get('max_overlap_pct')}]")

det = safe_load("reports/det.json")
if det:
    pdm = det.get("per_detector_metrics", {}).get("p_ensemble_fake", {})
    if pdm:
        print(f"Detector ensemble     : acc={pdm.get('accuracy')}%  F1(FAKE)={pdm.get('f1_fake')}%  on n={pdm.get('n')}")

# Pilot pack
import os
pilot_dir = "reports/pilot_pack"
if os.path.isdir(pilot_dir):
    files = os.listdir(pilot_dir)
    print(f"\nPilot pack outputs ({pilot_dir}/):")
    for f in sorted(files):
        sz = os.path.getsize(os.path.join(pilot_dir, f))
        print(f"  {f}  ({sz:,} bytes)")

print("\n" + "=" * 70)
print("Targets:")
print("  Pool size 668 (±15)        Indicator coverage  12/12")
print("  Faithfulness audit  100%    Pairwise overlap   <13% mean")
print("  Detector ensemble   100%    Pilot pack          3 files")
print("=" * 70)


## 18. Sample 6 cards from a player's deck

Quick sanity check — read 6 cards as a SHS student would. Do they make sense?


In [ ]:
import json, random

decks_dir = "generated/decks"
deck_files = sorted([f for f in os.listdir(decks_dir) if f.endswith(".json")])
if not deck_files:
    print("⚠ No deck files in generated/decks/")
else:
    deck = json.load(open(os.path.join(decks_dir, deck_files[0]), encoding="utf-8"))
    cards = deck.get("cards", deck if isinstance(deck, list) else [])
    print(f"Player: {deck_files[0].replace('.json','')}")
    print(f"Deck size: {len(cards)}")
    print()
    random.seed(42)
    sample = random.sample(cards, min(6, len(cards)))
    for i, c in enumerate(sample, 1):
        print(f"--- Card {i} | {c.get('verdict')} | {c.get('candidate')} | {c.get('provenance',{}).get('tactic')} | {c.get('provenance',{}).get('tier')} ---")
        print(f"  {c.get('text','')[:300]}")
        print()


## 19. Sample of the pilot pack (P1.2)

Print the first 2 cards from the questionnaire so you can see what students will see.


In [ ]:
quest_path = "reports/pilot_pack/questionnaire.md"
if os.path.exists(quest_path):
    text = open(quest_path, encoding="utf-8").read()
    # Print first 2 cards (header + Card 1 + Card 2)
    cards_split = text.split("---")
    print("\n---".join(cards_split[:5]))
else:
    print("⚠ Pilot pack not generated. Run cell 16.")


## 20. Save outputs (to Drive if mounted, otherwise download as zip)

Two paths:

- **If Drive is mounted:** copies `generated/` and `reports/` to `MyDrive/MINERVA_outputs/<timestamp>/` so you can keep them after the Colab session ends.
- **If Drive is NOT mounted:** bundles `generated/` and `reports/` into a single zip and triggers a browser download via `google.colab.files.download(...)`. No need to navigate the Files sidebar.

Pick whichever flow suits your runtime. The cell auto-detects.


In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

# v2.9.0: list of artifacts to bundle. We include the QLattice equation
# and structured metadata so reviewers can audit the symbolic-regression
# rule used for explainability and GPT-2 conditioning.
_QL_FILES = [
    "models/qlattice_equation.txt",
    "models/best_qlattice.json",
    "models/qlattice/best_qlattice.json",
]

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if DRIVE_MOUNT_OK:
    # Path A: Drive is mounted — copy directories to Drive
    target = Path(f"/content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}")
    target.mkdir(parents=True, exist_ok=True)

    for src in ["generated", "reports"]:
        if not Path(src).exists():
            continue
        dst = target / src
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✓ {src} → {dst}")

    # v2.9.0: bundle the QLattice equation files alongside the rest
    ql_target = target / "models"
    ql_target.mkdir(parents=True, exist_ok=True)
    for ql_file in _QL_FILES:
        if Path(ql_file).exists():
            dst = ql_target / Path(ql_file).name
            shutil.copy2(ql_file, dst)
            print(f"  ✓ {ql_file} → {dst}")
        else:
            print(f"  ⏭  {ql_file} not found")

    print(f"\n✓ All outputs saved to Drive: /content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}/")

else:
    # Path B: Drive not mounted — bundle into a single zip and trigger browser download
    print("⚠ Drive not mounted. Bundling outputs into a zip for download...")

    bundle_name = f"minerva_outputs_{ts}"
    bundle_root = Path(f"/tmp/{bundle_name}")
    bundle_root.mkdir(parents=True, exist_ok=True)

    for src in ["generated", "reports"]:
        if not Path(src).exists():
            print(f"  ⏭  {src}/ not found — skipping")
            continue
        shutil.copytree(src, bundle_root / src)
        print(f"  ✓ Staged {src}/")

    # v2.9.0: stage the QLattice equation files into models/ in the zip
    ql_target = bundle_root / "models"
    ql_target.mkdir(parents=True, exist_ok=True)
    for ql_file in _QL_FILES:
        if Path(ql_file).exists():
            shutil.copy2(ql_file, ql_target / Path(ql_file).name)
            print(f"  ✓ Staged {ql_file}")
        else:
            print(f"  ⏭  {ql_file} not found — skipping")

    # Zip the staged bundle
    zip_path_no_ext = f"/tmp/{bundle_name}"
    archive_path = shutil.make_archive(zip_path_no_ext, "zip", root_dir=str(bundle_root))
    archive_size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f"  ✓ Created {archive_path} ({archive_size_mb:.1f} MB)")

    # Trigger browser download (Colab only)
    try:
        from google.colab import files as _colab_files
        print(f"\n⬇  Triggering browser download of {Path(archive_path).name} ...")
        _colab_files.download(archive_path)
        print(f"\n✓ Download initiated. Check your browser's download tray.")
    except ImportError:
        # Not running in Colab — print the local path
        print(f"\n⚠  Not running in Google Colab — automatic download unavailable.")
        print(f"   The bundle is at: {archive_path}")
        print(f"   Outputs are also still at:")
        print(f"     /content/MINERVA/generated/")
        print(f"     /content/MINERVA/reports/")
        print(f"   Open the Files tab in Colab's left sidebar to download manually.")
    except Exception as e:
        # Network or other Colab download failure
        print(f"\n⚠  Download trigger failed: {e}")
        print(f"   Bundle is still available at: {archive_path}")
        print(f"   You can right-click it in the Files tab and select 'Download'.")


---

## What to do next

**For the demo:**
1. Open `reports/pilot_pack/printable_card_pack.html` in your browser
2. Ctrl+P → Save as PDF — you have a 30-card printable pack
3. Paste `reports/pilot_pack/questionnaire.md` into a Google Form (one section per card)
4. Hand to 5 SHS students; score against `reports/pilot_pack/answer_key.md`

**For thesis defense:**
- The verified metrics above answer the panel's "does the system work?" question
- 750 data points from the pre-pilot (5 students × 30 cards × 5 questions) is empirical evidence
- For "do detectors generalize beyond your templates?" run the held-out eval (HANDOFF.md P1.1)
- For "did anyone else verify your indicator labels?" run the IRR study (HANDOFF.md P1.3)

**For Unity integration:**
- The Unity client reads `generated/unity_cards_pool.json` (or per-user files in `generated/decks/`)
- Schema doc: `docs/V2.6_CHANGES.md` + `scripts/minerva_schemas.py`
- C# port of script 28 (deck draw) is HANDOFF.md P3.2

Pipeline runtime varies by Colab tier (Free / Pro / Pro+) — typically **~3 minutes** end-to-end.


## (Optional, run-once) Refresh the JCBlaise blocklist from the source dataset

This cell downloads the actual JCBlaise dataset from HuggingFace and extracts
every person-like name appearing 3+ times. The output replaces the seed
blocklist shipped with the repo.

**You only need to run this once** — the output (`templates/jcblaise_real_names_blocklist.txt`)
should be committed to the repo so all future pipeline runs use the same
dataset-derived list.

Skip this cell if the seed blocklist is sufficient for your demo. Run it if
you want the full empirically-grounded list.

Backed by Cruz, Tan, & Cheng (2020) LREC — the source dataset citation, and
Pilan et al. (2022) *Computational Linguistics* — corpus-derived blocklists
as the recommended complement to regex-based PII detection.


In [ ]:
# Optional — run once, then commit the output to your repo
import os

# Make sure datasets is installed (Colab usually has it)
!python -m pip install -q datasets 2>&1 | tail -1

# Run the extractor
!python scripts/34_extract_jcblaise_names.py \
    --out_file templates/jcblaise_real_names_blocklist.txt \
    --report_out reports/jcblaise_extraction.json \
    --min_count 3

# Show the report
print()
import json as _json
if os.path.exists("reports/jcblaise_extraction.json"):
    _r = _json.load(open("reports/jcblaise_extraction.json"))
    print(f"Rows processed     : {_r['rows_processed']:,}")
    print(f"Unique names       : {_r['unique_names_total']:,}")
    print(f"Names blocklisted  : {_r['names_blocklisted']:,}")
    print(f"Top 5 names in dataset:")
    for x in _r['top_50_names_overall'][:5]:
        print(f"  - {x['name']} ({x['count']:,} occurrences)")
